In [1]:
import gudhi as gd
import geopandas as gpd
import numpy as np

In [2]:
hvi_df = gpd.read_file("../data/CHAT-Los Angeles County-vulnerability-indicators.csv")
hvi_df.head()

,census_tract,census_county,census_city,heat_health_action_index,perc_children,perc_no_hs_diploma,perc_elderly,perc_outdoor_workers,tract_population,perc_poverty,...,perc_low_birth_weight,cardio_disease_prevalence,perc_ambulatory_disability,perc_cognitive_disability,pm25_concentration,perc_impervious_surfaces,change_in_dev,perc_no_tree_canopy,uhii_avgdeltat,ozone_exceedance
0,6037101110,Los Angeles County,Tujunga,36.53,4.90,19.70,13.40,6.57,4824,14.50,...,3.44,10.81,7.40,5.00,11.04,43.11,0.00,94.14,2.26,0.19
1,6037101122,Los Angeles County,Tujunga,19.76,1.70,4.70,15.50,7.68,3291,2.70,...,5.05,8.87,3.90,2.00,10.95,25.36,12.44,90.80,,0.19
2,6037101210,Los Angeles County,Tujunga,43.45,3.90,22.80,8.80,8.38,5882,24.00,...,4.89,10.81,8.20,5.20,11.11,56.99,0.00,95.99,,0.18
3,6037101220,Los Angeles County,Tujunga,40.68,4.70,17.80,11.80,4.80,2902,14.20,...,4.27,10.81,10.40,5.10,11.14,45.24,0.00,95.35,,0.18
4,6037101300,Los Angeles County,La Crescenta,23.03,2.50,10.10,20.20,8.01,4410,7.30,...,3.48,8.16,7.40,2.90,11.19,36.00,25.15,92.52,,0.17


In [3]:
hvi_df = hvi_df[["census_tract", "census_county","census_city", "heat_health_action_index"]].copy()
hvi_df = hvi_df.rename(columns={"heat_health_action_index": "hvi"})
hvi_df.dropna(subset=['hvi'], inplace=True)
hvi_df.head()

,census_tract,census_county,census_city,hvi
0,6037101110,Los Angeles County,Tujunga,36.53
1,6037101122,Los Angeles County,Tujunga,19.76
2,6037101210,Los Angeles County,Tujunga,43.45
3,6037101220,Los Angeles County,Tujunga,40.68
4,6037101300,Los Angeles County,La Crescenta,23.03


In [4]:
hvi_df.to_csv("../data/HVI_LACounty.csv", index=False)

In [5]:
svi_df = gpd.read_file("../data/California_2020_SVI.csv")
svi_df.head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,06,California,CA,06001,Alameda,06001400100,"Census Tract 4001, Alameda County, California",2.681809279414,3035,402,...,14.0,3.0,0.0,1.3,0.0,1.3,5.6,3.9,0.6,0.9
1,06,California,CA,06001,Alameda,06001400200,"Census Tract 4002, Alameda County, California",0.22647198912,1983,209,...,11.0,4.7,0.3,0.4,0.0,2.0,10.6,5.0,0.4,0.6
2,06,California,CA,06001,Alameda,06001400300,"Census Tract 4003, Alameda County, California",0.42889754568,5058,559,...,15.3,7.6,0.1,0.3,0.7,1.1,4.1,3.7,1.1,1.4
3,06,California,CA,06001,Alameda,06001400400,"Census Tract 4004, Alameda County, California",0.276502314076,4179,529,...,10.0,3.3,0.6,0.8,0.0,1.0,6.7,2.7,0.1,0.2
4,06,California,CA,06001,Alameda,06001400500,"Census Tract 4005, Alameda County, California",0.228349989248,4021,631,...,9.6,3.9,0.0,1.0,0.0,1.0,9.4,5.2,0.0,1.0


In [6]:
# only keep observations within LA county
svi = svi_df[svi_df['COUNTY'] == 'Los Angeles']

In [7]:
# reset index & drop old index
svi = svi_df.reset_index(drop = True)

In [8]:
# keep only LA county and drop unnecessary columns
svi = (
    svi_df[svi_df['COUNTY'] == 'Los Angeles']
    .drop(columns=[
        'ST',
        'STATE',
        'ST_ABBR',
        'STCNTY',
        'COUNTY',
        'AREA_SQMI'
    ])
    .copy()
)

# rename key column for later merge
svi = svi.rename(columns={'FIPS': 'census_tract'})
svi.head()

,census_tract,LOCATION,E_TOTPOP,M_TOTPOP,E_HU,M_HU,E_HH,M_HH,E_POV150,M_POV150,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
1378,06037101110,"Census Tract 1011.10, Los Angeles County, Cali...",3923,460,1629,93,1505,112,801,341,...,10.3,3.9,0.1,0.2,0.1,0.1,3.3,2.1,0.2,0.4
1379,06037101122,"Census Tract 1011.22, Los Angeles County, Cali...",4119,858,1406,144,1341,151,287,214,...,10.3,4.3,0.0,1.0,0.0,1.0,5.0,4.0,0.0,1.0
1380,06037101220,"Census Tract 1012.20, Los Angeles County, Cali...",3775,474,1462,208,1430,208,1017,310,...,10.3,6.0,0.0,1.1,0.0,1.1,2.8,2.0,0.2,0.2
1381,06037101221,"Census Tract 1012.21, Los Angeles County, Cali...",3787,651,1567,322,1513,325,1399,632,...,7.1,4.2,0.0,1.0,0.0,1.0,0.0,1.0,2.6,2.9
1382,06037101222,"Census Tract 1012.22, Los Angeles County, Cali...",2717,442,986,173,969,174,1507,486,...,2.4,5.1,0.0,1.5,0.0,1.5,0.0,1.5,0.0,1.5


## Merging SVI & HVI datasets using census tract

In [9]:
# merge dataframes on tract number
hvi_svi = hvi_df.merge(svi, how = 'outer', on = 'census_tract')

In [10]:
hvi_svi.head()

,census_tract,census_county,census_city,hvi,LOCATION,E_TOTPOP,M_TOTPOP,E_HU,M_HU,E_HH,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,06037101110,NaN,NaN,NaN,"Census Tract 1011.10, Los Angeles County, Cali...",3923,460,1629,93,1505,...,10.3,3.9,0.1,0.2,0.1,0.1,3.3,2.1,0.2,0.4
1,06037101122,NaN,NaN,NaN,"Census Tract 1011.22, Los Angeles County, Cali...",4119,858,1406,144,1341,...,10.3,4.3,0.0,1.0,0.0,1.0,5.0,4.0,0.0,1.0
2,06037101220,NaN,NaN,NaN,"Census Tract 1012.20, Los Angeles County, Cali...",3775,474,1462,208,1430,...,10.3,6.0,0.0,1.1,0.0,1.1,2.8,2.0,0.2,0.2
3,06037101221,NaN,NaN,NaN,"Census Tract 1012.21, Los Angeles County, Cali...",3787,651,1567,322,1513,...,7.1,4.2,0.0,1.0,0.0,1.0,0.0,1.0,2.6,2.9
4,06037101222,NaN,NaN,NaN,"Census Tract 1012.22, Los Angeles County, Cali...",2717,442,986,173,969,...,2.4,5.1,0.0,1.5,0.0,1.5,0.0,1.5,0.0,1.5
